In [1]:
import import_ipynb
from prolog import KnowledgeBase, Term, parse_term, parse_goals, split_args
import re
from typing import List, Dict, Optional

importing Jupyter notebook from prolog.ipynb


In [2]:
# --- Extend KB with safe retract for full clauses ---
def retract_clause(self, clause: str):
    """
    Retract a clause (fact or rule) exactly matching head and body.
    """
    clause = clause.strip().rstrip(".")
    if ":-" in clause:
        head_str, body_str = clause.split(":-")
        head = parse_term(head_str.strip())
        body = [parse_term(b.strip()) for b in split_args(body_str)]
    else:
        head = parse_term(clause)
        body = []
    # Remove clause that matches exactly
    self.clauses = [(h, b) for (h, b) in self.clauses if not (h == head and b == body)]

# Add to KnowledgeBase
KnowledgeBase.retract_clause = retract_clause

# --- ILP System ---
from itertools import product
from typing import List

class SimpleILP:
    def __init__(self, kb: KnowledgeBase):
        self.kb = kb

    def learn(self, target: str, pos_examples: List[str], neg_examples: List[str], max_body_len=2, verbose=False):
        """
        Learn a single Horn clause with variables that covers positive examples
        and excludes negative examples.
        """
        target_term = parse_term(target)

        # Generate candidate body literals using KB predicates
        body_predicates = [h.name for (h, _) in self.kb.clauses]
        var_names = ["X", "Y", "Z", "W", "U", "V"]  # reusable variable names
        candidate_bodies = []

        for r in range(1, max_body_len + 1):  # body length 1 or 2
            for preds in product(body_predicates, repeat=r):
                arities = [self._get_arity(p) for p in preds]
                # Assign variables to each argument
                var_combinations = [list(product(var_names, repeat=a)) for a in arities]
                for args_tuple in product(*var_combinations):
                    body_terms = []
                    for i, pred in enumerate(preds):
                        body_terms.append(Term(pred, [Term(a) for a in args_tuple[i]]))
                    candidate_bodies.append(body_terms)

        if verbose:
            print(f"Generated {len(candidate_bodies)} candidate bodies with variables.")

        # Test each candidate
        for body in candidate_bodies:
            clause_str = f"{target_term} :- {', '.join(map(str, body))}."
            self.kb.assertz(clause_str)
            pos_ok = all(self.kb.query(ex) for ex in pos_examples)
            neg_ok = all(not self.kb.query(ex) for ex in neg_examples)
            if pos_ok and neg_ok:
                if verbose:
                    print(f"Learned clause: {clause_str}")
                return clause_str
            self.kb.retract_clause(clause_str)

        if verbose:
            print("No consistent clause found.")
        return None

    def _get_arity(self, pred_name: str):
        for h, _ in self.kb.clauses:
            if h.name == pred_name:
                return len(h.args)
        return 1


In [3]:
from itertools import product
from typing import List, Set

class SimpleILPOptimized(SimpleILP):
    def learn_fast(self, target: str, pos_examples: List[str], neg_examples: List[str],
                   max_body_len=2, verbose=False, max_depth=20):
        """
        Optimized ILP: variable-based, avoids recursion loops, faster candidate generation.
        """
        target_term = parse_term(target)

        # Collect relevant predicates from KB and positive examples
        relevant_preds: Set[str] = set(h.name for h, _ in self.kb.clauses)
        for ex in pos_examples:
            for t in parse_goals(ex):
                relevant_preds.update(self._collect_predicates(t))

        if verbose:
            print(f"Relevant predicates: {relevant_preds}")

        var_names = ["X", "Y", "Z", "W", "U", "V"]

        # Generate candidate bodies
        for body_len in range(1, max_body_len + 1):
            for preds in product(relevant_preds, repeat=body_len):
                arities = [self._get_arity(p) for p in preds]

                # Generate variable assignments for each predicate
                var_combinations = [list(product(var_names, repeat=a)) for a in arities]

                for args_tuple in product(*var_combinations):
                    body_terms = []
                    used_vars_global = set()
                    valid = True
                    for i, pred in enumerate(preds):
                        args = args_tuple[i]
                        # Skip if duplicate vars in same literal or used globally
                        if len(set(args)) < len(args) or any(v in used_vars_global for v in args):
                            valid = False
                            break
                        used_vars_global.update(args)
                        body_terms.append(Term(pred, [Term(v) for v in args]))
                    if not valid:
                        continue

                    clause_str = f"{target_term} :- {', '.join(map(str, body_terms))}."
                    self.kb.assertz(clause_str)

                    # Evaluate examples with limited recursion depth
                    pos_ok = all(self.kb.query(ex, verbose=False) for ex in pos_examples)
                    neg_ok = all(not self.kb.query(ex, verbose=False) for ex in neg_examples)

                    if pos_ok and neg_ok:
                        if verbose:
                            print(f"Learned clause: {clause_str}")
                        return clause_str

                    self.kb.retract_clause(clause_str)

        if verbose:
            print("No consistent clause found in fast learning.")
        return None

    def _collect_predicates(self, term: Term) -> Set[str]:
        preds = set()
        if term.is_compound():
            preds.add(term.name)
            for arg in term.args:
                preds.update(self._collect_predicates(arg))
        return preds


In [17]:
kb = KnowledgeBase()
kb.assertz("parent(alice, bob).")
kb.assertz("parent(bob, charlie).")
# kb.assertz("female(alice).")

ilp = SimpleILP(kb)

pos = ["grandparent(alice, charlie)."]
neg = ["grandparent(bob, charlie)."]

clause = ilp.learn("grandparent(X, Y)", pos, neg, verbose=True)
print("Learned clause:", clause)


Generated 5256 candidate bodies with variables.
Learned clause: grandparent(X, Y) :- parent(X, Z), parent(Z, Y).
Learned clause: grandparent(X, Y) :- parent(X, Z), parent(Z, Y).


In [11]:
kb = KnowledgeBase()
kb.assertz("above(dear, delhi).")
kb.assertz("rightof(dear, sir).")
kb.assertz("above(hello,dear)")
kb.assertz("rightof(delhi, 110001)")
kb.assertz("rightof(hello,how)")

In [12]:
pos = ["pincode(11001)"]
neg = ["pincode(dear)"]

In [15]:
ilp = SimpleILP(kb)

In [16]:
clause = ilp.learn("pincode(X)", pos, neg, verbose=True)
print("Learned clause:", clause)

Generated 32580 candidate bodies with variables.
No consistent clause found.
Learned clause: None
